TASK 3.1 Features, split and baseline

In [ ]:
import pandas  as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, roc_auc_score, recall_score, confusion_matrix, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

In [233]:
df=pd.read_csv("appointments_clean.csv")
df

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
3,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
4,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
985,A0996,P415,C05,2025-01-09,2025-01-31,20,M,13.0,700,Phone,Consultation,No,1,Attended,22,0
986,A0997,P247,C05,2025-02-23,2025-02-27,77,M,17.4,400,App,Follow-up,Yes,3,No-Show,4,1
987,A0998,P422,C06,2025-05-05,2025-05-16,58,M,3.2,850,Walk-In,Consultation,No,1,Attended,11,0
988,A0999,P509,C05,2025-03-30,2025-04-12,59,F,26.0,300,Walk-In,Follow-up,Yes,0,Attended,13,0


In [234]:
df['reminder_sent']=df['reminder_sent'].map({"Yes":1, "No":0})
df

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,1,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,0,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,0,0,Attended,6,0
3,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,1,0,Attended,4,0
4,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,0,0,No-Show,40,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
985,A0996,P415,C05,2025-01-09,2025-01-31,20,M,13.0,700,Phone,Consultation,0,1,Attended,22,0
986,A0997,P247,C05,2025-02-23,2025-02-27,77,M,17.4,400,App,Follow-up,1,3,No-Show,4,1
987,A0998,P422,C06,2025-05-05,2025-05-16,58,M,3.2,850,Walk-In,Consultation,0,1,Attended,11,0
988,A0999,P509,C05,2025-03-30,2025-04-12,59,F,26.0,300,Walk-In,Follow-up,1,0,Attended,13,0


In [ ]:
X=df.drop(columns=["no_show", "gender", "patient_id", "appointment_id", "booked_on", "appt_date", "clinic_id", "status"])
y=df["no_show"]

,age,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,lead_days
0,60,18.0,850,Phone,Consultation,1,1,1
1,39,12.5,350,Walk-In,Follow-up,0,2,10
2,63,4.1,850,Walk-In,Consultation,0,0,6
3,52,7.5,750,App,Consultation,1,0,4
4,64,19.0,700,Phone,Consultation,0,0,40
...,...,...,...,...,...,...,...,...
985,20,13.0,700,Phone,Consultation,0,1,22
986,77,17.4,400,App,Follow-up,1,3,4
987,58,3.2,850,Walk-In,Consultation,0,1,11
988,59,26.0,300,Walk-In,Follow-up,1,0,13


In [258]:
X_encoded = pd.get_dummies(
    X[["booking_channel", "visit_type"]], dtype=int)

In [237]:
X_encoded
X_final=pd.concat([X.drop(columns=["booking_channel", "visit_type"]),  X_encoded], axis=1)
X_final

,age,distance_km,fee_inr,reminder_sent,prior_no_shows,lead_days,booking_channel_App,booking_channel_Phone,booking_channel_Walk-In,visit_type_Consultation,visit_type_Follow-up
0,60,18.0,850,1,1,1,0,1,0,1,0
1,39,12.5,350,0,2,10,0,0,1,0,1
2,63,4.1,850,0,0,6,0,0,1,1,0
3,52,7.5,750,1,0,4,1,0,0,1,0
4,64,19.0,700,0,0,40,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...
985,20,13.0,700,0,1,22,0,1,0,1,0
986,77,17.4,400,1,3,4,1,0,0,0,1
987,58,3.2,850,0,1,11,0,0,1,1,0
988,59,26.0,300,1,0,13,0,0,1,0,1


In [261]:
X_train, X_test, y_train, y_test=train_test_split(X_final, y, test_size=0.2, random_state=42)
print("Training size:", len(X_train), "\n")
print("Test size:", len(X_test), "\n")
print("No-shows in test set:", y_test.sum(), "\n")
print(f"Baseline accuracy (always Attended): {(y_test == 0).mean():.2%}", "\n")

X_test

Training size: 792 

Test size: 198 

No-shows in test set: 49 

Baseline accuracy (always Attended): 75.25% 



,age,distance_km,fee_inr,reminder_sent,prior_no_shows,lead_days,booking_channel_App,booking_channel_Phone,booking_channel_Walk-In,visit_type_Consultation,visit_type_Follow-up
213,54,16.1,400,1,0,2,1,0,0,0,1
331,64,1.8,750,0,1,3,1,0,0,1,0
501,30,11.4,800,0,1,6,0,1,0,1,0
309,31,15.8,500,0,0,3,0,1,0,0,1
88,71,13.9,750,1,0,25,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...
450,66,26.0,350,0,2,13,0,0,1,0,1
705,51,11.8,1000,1,0,11,1,0,0,1,0
305,44,4.4,550,0,0,8,0,1,0,0,1
807,59,5.2,700,0,0,22,0,1,0,1,0


TASK 3.2 Logistic regression

In [239]:
X_final.dtypes

age                          int64
distance_km                float64
fee_inr                      int64
reminder_sent                int64
prior_no_shows               int64
lead_days                    int64
booking_channel_App          int64
booking_channel_Phone        int64
booking_channel_Walk-In      int64
visit_type_Consultation      int64
visit_type_Follow-up         int64
dtype: object

In [240]:
scaler= StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [241]:
lr=LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr=lr.predict(X_test)

c:\Stock Sense\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [242]:
print("Accuracy= ", accuracy_score(y_test, y_pred_lr))
print("Precision= ", precision_score(y_test, y_pred_lr))
print("Recall= ", recall_score(y_test, y_pred_lr))
print("F1 Score= ", f1_score(y_test, y_pred_lr))
print("ROC AUC= ", roc_auc_score(y_test, y_pred_lr))
print("Confusion Matrix= \n", confusion_matrix(y_test, y_pred_lr))

Accuracy=  0.7878787878787878
Precision=  0.6296296296296297
Recall=  0.3469387755102041
F1 Score=  0.4473684210526316
ROC AUC=  0.6399123407752363
Confusion Matrix= 
 [[139  10]
 [ 32  17]]


In [243]:
for feature, coef in zip(X_final.columns, lr.coef_[0]):
    print(f"{feature}: {coef:.2f}")

age: -0.02
distance_km: 0.04
fee_inr: 0.00
reminder_sent: -1.13
prior_no_shows: 0.67
lead_days: 0.07
booking_channel_App: -0.14
booking_channel_Phone: 0.40
booking_channel_Walk-In: -0.43
visit_type_Consultation: -0.08
visit_type_Follow-up: -0.09


In [244]:
idx = list(X_final.columns).index("reminder_sent")
coef_reminder = lr.coef_[0][idx]
scale_reminder = scaler.scale_[idx]

exp_coef = np.exp(coef_reminder)
exp_unscaled = np.exp(coef_reminder / scale_reminder)

print(f"\nreminder_sent: {exp_coef:.2f}")
print(f"reminder_sent np.exp(coefficient / scaler.scale_): {exp_unscaled:.2f}")

print("\nWhy they differ: np.exp(coefficient) gives the odds multiplier for a 1-standard-deviation change on the standardized feature,"
      f"whereas dividing by scaler.scale_ before exponentiating scales the effect to a full 1-unit (0 to 1) change on the original indicator.")

print(f"Conclusion: In this model, holding the other features constant, reminder_sent going from No (0) to Yes (1)"
      f"is associated with a {exp_unscaled:.2f}x change in no-show odds, though because the data are observational, this does not prove that reminders cause fewer no-shows.")


reminder_sent: 0.32
reminder_sent np.exp(coefficient / scaler.scale_): 0.10

Why they differ: np.exp(coefficient) gives the odds multiplier for a 1-standard-deviation change on the standardized feature,whereas dividing by scaler.scale_ before exponentiating scales the effect to a full 1-unit (0 to 1) change on the original indicator.
Conclusion: In this model, holding the other features constant, reminder_sent going from No (0) to Yes (1)is associated with a 0.10x change in no-show odds, though because the data are observational, this does not prove that reminders cause fewer no-shows.


TASK 3.3 Decision tree and overfitting

In [245]:
dtree=DecisionTreeClassifier(random_state=42)
dtree.fit(X_train, y_train)
y_pred_tree=dtree.predict(X_test)

print(f"No depth limit -> Train acc: {dtree.score(X_train, y_train):.2f}")
print(f"Test acc: {dtree.score(X_test, y_test):.2f}")

No depth limit -> Train acc: 1.00
Test acc: 0.68


In [246]:
dtree3=DecisionTreeClassifier(max_depth=3, random_state=42)
dtree3.fit(X_train, y_train)
y_pred_tree2=dtree3.predict(X_test)

print(f"No depth limit -> Train acc: {dtree3.score(X_train, y_train):.2f}")
print(f"Test acc: {dtree3.score(X_test, y_test):.2f}")

No depth limit -> Train acc: 0.82
Test acc: 0.78


In [247]:
print("What the gap shows is the large train–test gap with no depth limit shows severe overfitting," \
"whereas constraining the tree to max_depth=3 sharply reduces the gap and improves generalization on unseen test data")

What the gap shows is the large train–test gap with no depth limit shows severe overfitting,whereas constraining the tree to max_depth=3 sharply reduces the gap and improves generalization on unseen test data


TASK 3.4 Random forest and a comparison table

In [265]:
rf=RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf=rf.predict(X_test)
y_pred_proba=rf.predict_proba(X_test)[:,1]

print(f"No depth limit -> Train acc: {rf.score(X_train, y_train):.2f}")
print(f"Test acc: {rf.score(X_test, y_test):.2f}")
print(f"Accuracy= {accuracy_score(y_test, y_pred_rf):.2f}")
print(f"Recall= {recall_score(y_test, y_pred_rf):.2f}")
print(f"ROC AUC= {roc_auc_score(y_test, y_pred_proba):.2f}")



No depth limit -> Train acc: 1.00
Test acc: 0.75
Accuracy= 0.75
Recall= 0.29
ROC AUC= 0.70


In [266]:
imp=pd.Series(rf.feature_importances_, index=X_final.columns).sort_values(ascending=False)
print(f"\nTop 3 feature importances:\n{imp.head(3)}")


Top 3 feature importances:
lead_days      0.222765
age            0.202164
distance_km    0.192075
dtype: float64


In [267]:
comparison_df = pd.DataFrame(
    {
        "Model": [
            "Logistic Regression",
            "Decision Tree (max_depth=3)",
            "Random Forest",
        ],
        "Accuracy": [
            accuracy_score(y_test, y_pred_lr),
            accuracy_score(y_test, y_pred_tree2),
            accuracy_score(y_test, y_pred_rf),
        ],
        "Recall": [
            recall_score(y_test, y_pred_lr),
            recall_score(y_test, y_pred_tree2),
            recall_score(y_test, y_pred_rf),
        ],
        "ROC-AUC": [
            roc_auc_score(y_test, y_pred_lr),
            roc_auc_score(y_test, y_pred_tree2),
            roc_auc_score(y_test, y_pred_proba),
        ],
    }
)

print("\nModel Comparison Table:\n")
print(comparison_df.to_string(index=False))


Model Comparison Table:

                      Model  Accuracy   Recall  ROC-AUC
        Logistic Regression  0.787879 0.346939 0.639912
Decision Tree (max_depth=3)  0.777778 0.224490 0.592111
              Random Forest  0.747475 0.285714 0.702301


TASK 3.5 Tune, choose the model and the threshold, then test once

In [ ]:
params={
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8]
    }
    
grid= GridSearchCV(RandomForestClassifier(random_state=42), params, cv=5, scoring="roc_auc")
grid.fit(X_train, y_train)

best_params=grid.best_params_
best_model=grid.best_estimator_

print("Best Parameters=", best_params)
print("Best Model=", best_model)

Best Parameters= {'max_depth': 5, 'n_estimators': 100}
Best Model= RandomForestClassifier(max_depth=5, random_state=42)


In [252]:
lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
dtree_pipe= DecisionTreeClassifier(max_depth=3, random_state=42)
rf_pipe= RandomForestClassifier(random_state=42)

cv_lr = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_dt3 = cross_val_score(dtree_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_rf = cross_val_score(rf_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_rf_tuned=grid.best_score_

In [253]:
print(f"CV ROC-AUC - Logistic Regression: {cv_lr:.2f}")
print(f"CV ROC-AUC - Depth-3 Tree:       {cv_dt3:.2f}")
print(f"CV ROC-AUC - Random Forest (100):{cv_rf:.2f}")
print(f"CV ROC-AUC - Tuned RF:           {grid.best_score_:.2f}")

CV ROC-AUC - Logistic Regression: 0.77
CV ROC-AUC - Depth-3 Tree:       0.68
CV ROC-AUC - Random Forest (100):0.74
CV ROC-AUC - Tuned RF:           0.77


In [254]:
oof_probs = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]

for t in [0.5, 0.35]:
    preds = (oof_probs >= t).astype(int)
    print(
        f"Threshold {t:.2f} -> "
        f"Precision: {precision_score(y_train, preds):.2f}, "
        f"Recall: {recall_score(y_train, preds):.2f}, "
        f"F1: {f1_score(y_train, preds):.2f}, "
        f"Flagged: {preds.sum()}"
    )

chosen_threshold = 0.35

#recall matters here because a missed warning costs an empty appointment slot whereas an extra call costs only a minute, 
# higher recall is valuable for catching more likely no-shows before they miss their visit

Threshold 0.50 -> Precision: 0.58, Recall: 0.25, F1: 0.35, Flagged: 73
Threshold 0.35 -> Precision: 0.51, Recall: 0.47, F1: 0.49, Flagged: 154


In [255]:
lr_pipe.fit(X_train, y_train)
test_probs = lr_pipe.predict_proba(X_test)[:, 1]
test_flags = (test_probs >= chosen_threshold).astype(int)

print(
    f"\nTest at {chosen_threshold:.2f} -> "
    f"Precision: {precision_score(y_test, test_flags):.2f}, "
    f"Recall: {recall_score(y_test, test_flags):.2f}, "
    f"Flagged: {test_flags.sum()}, "
    f"ROC-AUC: {roc_auc_score(y_test, test_probs):.2f}"
)




Test at 0.35 -> Precision: 0.57, Recall: 0.57, Flagged: 49, ROC-AUC: 0.79


In [256]:
test_risk_scores = pd.DataFrame(
    {
        "appointment_id": df.loc[X_test.index, "appointment_id"].values,
        "no_show": y_test.values,
        "risk_prob": test_probs,
        "flag": test_flags,
    }
)
test_risk_scores.to_csv("test_risk_scores.csv", index=False)

#A clinic would choose the threshold, by weighing how many follow-up calls the front desk can make in an evening
# for the cost of an empty appointment slot.